In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class OxidizeAlcohol(MorphingOperator):
    def __init__(self):
        super(OxidizeAlcohol, self).__init__()
        self._name = "Oxidize Alcohol"
        self._target_atoms = [] 
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6X4;H1,H2]")

    def setOriginal(self, mol):
        super(OxidizeAlcohol, self).setOriginal(mol)
        self._target_atoms = []
        
        if not self.original:
            return
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return
            
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
        
            self._target_atoms.append((match[0], match[1]))

    def morph(self):
        if not self.original: 
            return None
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return None
            
        if not self._target_atoms:
            return MolpherMol(other=rdkit_mol)
            
        idx_o, idx_c = random.choice(self._target_atoms)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            if rw_mol.GetBondBetweenAtoms(idx_o, idx_c):
                rw_mol.RemoveBond(idx_o, idx_c)
            rw_mol.AddBond(idx_o, idx_c, Chem.BondType.DOUBLE)
            
            new_mol = rw_mol.GetMol()
    
            for idx in [idx_o, idx_c]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except Exception as e:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name

class OxidizeAldehydeToAcid(MorphingOperator):
    def __init__(self):
        super(OxidizeAldehydeToAcid, self).__init__()
        self._name = "Oxidize Aldehyde to Acid"
        self._target_carbons = [] 
        self.PATTERN = Chem.MolFromSmarts("[CX3H1](=O)[#6,#1]")

    def setOriginal(self, mol):
        super(OxidizeAldehydeToAcid, self).setOriginal(mol)
        self._target_carbons = []
        
        if not self.original:
            return
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return
            
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_carbons.append(match[0])

    def morph(self):
        if not self.original: 
            return None
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return None
        
        if not self._target_carbons:
            return MolpherMol(other=rdkit_mol)
            
        idx_c = random.choice(self._target_carbons)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            new_o_idx = rw_mol.AddAtom(Chem.Atom(8))
            
            rw_mol.AddBond(idx_c, new_o_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [idx_c, new_o_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
        
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except Exception as e:
            return MolpherMol(other=rdkit_mol)
        
    def getName(self):
        return self._name

oxidize_alcohol_op = OxidizeAlcohol()
oxidize_aldehyde_op = OxidizeAldehydeToAcid() 

start_mol = MolpherMol("CCCO")          # 1-Προπανόλη
target_mol = MolpherMol("CCC(=O)O")     # Προπιονικό οξύ
tree = ETree.create(source=start_mol, target=target_mol)
tree.morphing_operators = (oxidize_alcohol_op, oxidize_aldehyde_op)

class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target

closest_info = FindClosest()
max_generations = 5

print("=== STARTING OXIDATION PATHWAY SEARCH ===")
while not tree.path_found and tree.generation_count < max_generations:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
    
    print('Generation #', tree.generation_count, sep='')
    print('Molecules in tree:', tree.mol_count)
    if closest_info.closest_mol:
        print('Closest molecule to target: {0} (Tanimoto distance: {1})'.format(
            closest_info.closest_mol.getSMILES(), closest_info.closest_distance))
    print("-" * 40)

if tree.path_found:
    print("SUCCESS: Το Molpher βρήκε το μονοπάτι οξείδωσης!")
else:
    print("FAILED: Δεν βρέθηκε το μονοπάτι.")
print("=========================================")

=== STARTING OXIDATION PATHWAY SEARCH ===
Generation #1
Molecules in tree: 2
Closest molecule to target: CCC=O (Tanimoto distance: 0.7333333333333334)
----------------------------------------
Generation #2
Molecules in tree: 3
Closest molecule to target: CCC(=O)O (Tanimoto distance: 0.0)
----------------------------------------
SUCCESS: Το Molpher βρήκε το μονοπάτι οξείδωσης!


In [2]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class OxidizeAlcohol(MorphingOperator):
    def __init__(self):
        super(OxidizeAlcohol, self).__init__()
        self._name = "Oxidize Alcohol"
        self._target_atoms = [] 
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6X4;H1,H2]")

    def setOriginal(self, mol):
        super(OxidizeAlcohol, self).setOriginal(mol)
        self._target_atoms = []
        
        if not self.original:
            return
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return
            
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
        
            self._target_atoms.append((match[0], match[1]))

    def morph(self):
        if not self.original: 
            return None
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return None
            
        if not self._target_atoms:
            return MolpherMol(other=rdkit_mol)
            
        idx_o, idx_c = random.choice(self._target_atoms)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            if rw_mol.GetBondBetweenAtoms(idx_o, idx_c):
                rw_mol.RemoveBond(idx_o, idx_c)
            rw_mol.AddBond(idx_o, idx_c, Chem.BondType.DOUBLE)
            
            new_mol = rw_mol.GetMol()
    
            for idx in [idx_o, idx_c]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except Exception as e:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name

class OxidizeAldehydeToAcid(MorphingOperator):
    def __init__(self):
        super(OxidizeAldehydeToAcid, self).__init__()
        self._name = "Oxidize Aldehyde to Acid"
        self._target_carbons = [] 
        self.PATTERN = Chem.MolFromSmarts("[CX3H1](=O)[#6,#1]")

    def setOriginal(self, mol):
        super(OxidizeAldehydeToAcid, self).setOriginal(mol)
        self._target_carbons = []
        
        if not self.original:
            return
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return
            
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_carbons.append(match[0])

    def morph(self):
        if not self.original: 
            return None
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return None
        
        if not self._target_carbons:
            return MolpherMol(other=rdkit_mol)
            
        idx_c = random.choice(self._target_carbons)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            new_o_idx = rw_mol.AddAtom(Chem.Atom(8))
            
            rw_mol.AddBond(idx_c, new_o_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [idx_c, new_o_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
        
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except Exception as e:
            return MolpherMol(other=rdkit_mol)
        
    def getName(self):
        return self._name

oxidize_alcohol_op = OxidizeAlcohol()
oxidize_aldehyde_op = OxidizeAldehydeToAcid() 

start_mol  = MolpherMol("CCCCCO")      # 1-πεντανόλη (5C) 
target_mol = MolpherMol("CCCCC(=O)O")  # πεντανοϊκό οξύ (5C)  
tree = ETree.create(source=start_mol, target=target_mol)
tree.morphing_operators = (oxidize_alcohol_op, oxidize_aldehyde_op)

class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target

closest_info = FindClosest()
max_generations = 5

print("--- STARTING MOLPHER SEARCH TREE ---")
max_generations = 40
while not tree.path_found and tree.generation_count < max_generations:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
    
    print(f"Generation #{tree.generation_count}")
    print(f"Molecules in tree: {tree.mol_count}")
    if closest_info.closest_mol:
        print(f"Closest to target: {closest_info.closest_mol.getSMILES()} (Distance: {closest_info.closest_distance:.4f})")
    print("-" * 40)

--- STARTING MOLPHER SEARCH TREE ---
Generation #1
Molecules in tree: 2
Closest to target: CCCCC=O (Distance: 0.6818)
----------------------------------------
Generation #2
Molecules in tree: 3
Closest to target: CCCCC(=O)O (Distance: 0.0000)
----------------------------------------
